In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("Advertising.csv")
dfs = df.iloc[:20,:]
dfs

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9
5,6,8.7,48.9,75.0,7.2
6,7,57.5,32.8,23.5,11.8
7,8,120.2,19.6,11.6,13.2
8,9,8.6,2.1,1.0,4.8
9,10,199.8,2.6,21.2,10.6


In [15]:
tv = dfs.iloc[:,1]
radio = dfs.iloc[:,2]
news = dfs.iloc[:,3]
sales = dfs.iloc[:,4]

In [17]:
# Standardize data by z-score
tv_standard = (tv - tv.mean()) / tv.std()
radio_standard = (radio - radio.mean()) / radio.std()
news_standard = (news - news.mean()) / news.std()
sales_standard = (sales - sales.mean()) / sales.std()

In [33]:
# Objective function is the mean squared error
# f(m, b) = (1/n) * (sum (( m * x[i] + b) - y[i]) ** 2 )
# Substitute standardized data into mean squared error
def f(m1, m2, m3, b):
    sales_pred = m1 * tv_standard + m2 * radio_standard + m3 * news_standard + b
    return ((sales_pred - sales_standard) ** 2).mean()

    #return ((m1 * tv_standard + b - sales_standard) ** 2 +
    #(m2 * radio_standard + b - sales_standard) ** 2 + 
    #(m3 * news_standard + b - sales_standard) ** 2).mean()

print(f(0, 0, 0, 0))

0.9500000000000002


In [34]:
# Gradient: Partial derivative
def f1(m1, m2, m3, b):

    sales_pred = m1 * tv_standard + m2 * radio_standard + m3 * news_standard + b
    error = sales_pred - sales_standard
    
    # Partial derivatives
    grad_m1 = 2 * (error * tv_standard).mean()
    grad_m2 = 2 * (error * radio_standard).mean()
    grad_m3 = 2 * (error * news_standard).mean()
    grad_b = 2 * error.mean()
    
    #grad_m1 = 2 * ((m1 * tv_standard + b - sales_standard) * tv_standard).mean()
    #grad_m2 = 2 * ((m2 * radio_standard + b - sales_standard) * radio_standard).mean()
    #grad_m3 = 2 * ((m3 * news_standard + b - sales_standard) * news_standard).mean()
    #grad_b = 2 * ((m1 * tv_standard + b - sales_standard) ** 2 + (m2 * radio_standard + b - sales_standard) ** 2 + (m3 * news_standard + b - sales_standard)).mean()

    return grad_m1, grad_m2, grad_m3, grad_b

f1(0, 0, 0, 0)

(np.float64(-1.6395571500810813),
 np.float64(-0.8162598518330719),
 np.float64(-0.3886107126867289),
 np.float64(4.4131365228849973e-16))

In [38]:
# Gradient descent
m10, m20, m30, b0 = 0, 0, 0, 0
alpha = 0.1

for i in range(5000):
    gradient_m1, gradient_m2, gradient_m3, gradient_b = f1(m10, m20, m30, b0) # partial derivative

    m10 = m10 - alpha * gradient_m1
    m20 = m20 - alpha * gradient_m2
    m30 = m30 - alpha * gradient_m3
    b0 = b0 - alpha * gradient_b

    if i % 500 == 0:
        print("Iteration {i} - Mean squared error: ", f(m10, m20, m30, b0))

Iteration {i} - Mean squared error:  0.6364814027268598
Iteration {i} - Mean squared error:  0.06852898782784961
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964
Iteration {i} - Mean squared error:  0.06852898782784964


In [44]:
# Reverse engineer to get true values of m1, m2, m3, b
m1 = (m10 * sales.std()) / tv.std()
m2 = (m20 * sales.std()) / radio.std()
m3 = (m30 * sales.std()) / news.std()
b = sales.mean() + sales.std() * b0 - ((m10 * sales.std() * tv.mean()) / tv.std()) - ((m20 * sales.std() * radio.mean()) / radio.std()) - ((m30 * sales.std() * news.mean()) / news.std())

print("m1: ", m1,
      "\nm2: ", m2, 
      "\nm3: ", m3, 
      "\nb: ", b)

m1:  0.055244099951379994 
m2:  0.16885567654598685 
m3:  -0.015225252984652996 
b:  2.8593828453004493


In [45]:
# Check with minimize function
from scipy.optimize import minimize

# Objective function: Mean squared error
def fMin(w):
    m1, m2, m3, b = w
    sales_pred = (m1 * tv) + (m2 * radio) + (m3 * news) + b
    return ((sales_pred - sales) ** 2).mean()

# Gradient: Partial derivatives
def f1Min(w):
    m1, m2, m3, b = w
    gradient_m1 = 2 * ((m1 * tv + b - sales) * tv).mean()
    gradient_m2 = 2 * ((m2 * radio + b - sales) * radio).mean()
    gradient_m3 = 2 * ((m3 * news + b - sales) * news).mean()
    gradient_b = 2 * (sales_pred - sales).mean()

    return m1, m2, m3, b

res = minimize(fMin, x0 = [0, 0, 0, 0])
res.x

array([ 0.05524408,  0.16885569, -0.01522528,  2.85938665])